# AI-Powered Document Intelligence for Healthcare

**Turn unstructured clinical documents into searchable, queryable data — entirely inside Snowflake.**

This notebook is the hands-on companion to the webinar. It walks through the full pipeline:

1. **Stage & ingest** clinical PDFs into a governed Snowflake stage
2. **Parse** them with `AI_PARSE_DOCUMENT` (full layout understanding)
3. **Search** them semantically with a Cortex Search service
4. **Ask** natural-language questions via a Snowflake Intelligence / CoWork agent (UI)
5. **Automate** the whole loop with a scheduled task

> All sample documents are **100% synthetic**. No real patient data.

---

### Prerequisites
- A role with `CREATE DATABASE`, `CREATE WAREHOUSE`, and the **`SNOWFLAKE.CORTEX_USER`** database role (required for `AI_PARSE_DOCUMENT`, `AI_EXTRACT`, and Cortex Search).
- For the agent step (Segment 4): `CREATE AGENT` privilege (e.g., via `SNOWFLAKE_INTELLIGENCE_ADMIN`).
- Update the **GitHub repo URL** in the Git-integration cell (Segment 1) to point at your fork.

## Segment 1 — Stage & Ingest

A **stage** is a governed landing zone for files — a folder that lives inside your security perimeter.
We create the database, schema, and a small auto-suspending warehouse, then a stage with a
**directory table** (`DIRECTORY = ( ENABLE = TRUE )`) and server-side encryption.

In [ ]:
%%sql -r dataframe_1
-- Database, schema, and a small auto-suspending warehouse
CREATE DATABASE IF NOT EXISTS HEALTHCARE_DOC_AI;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_DOC_AI.CLINICAL_DOCS;

CREATE WAREHOUSE IF NOT EXISTS HEALTHCARE_WH
  WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE DATABASE HEALTHCARE_DOC_AI;
USE SCHEMA CLINICAL_DOCS;
USE WAREHOUSE HEALTHCARE_WH;

In [ ]:
%%sql -r dataframe_2
-- Directory table keeps an automatic inventory of files; SSE encrypts at rest
CREATE STAGE IF NOT EXISTS CLINICAL_DOCS_STAGE
  DIRECTORY = ( ENABLE = TRUE )
  ENCRYPTION = ( TYPE = 'SNOWFLAKE_SSE' );

### Load the sample PDFs via Git integration (reproducible path)

This pulls the six sample documents straight from the GitHub repo into the stage — no manual upload.

> **Fallback:** if Git integration isn't enabled in your tenant, skip this cell and upload the
> six PDFs from `/sample_documents/` into `CLINICAL_DOCS_STAGE` via the Snowsight UI
> (Data → Databases → HEALTHCARE_DOC_AI → CLINICAL_DOCS → Stages → CLINICAL_DOCS_STAGE → + Files).

In [ ]:
%%sql -r dataframe_3
-- 1) API integration for GitHub
CREATE OR REPLACE API INTEGRATION github_api_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-aprah')
  ENABLED = TRUE;

-- 2) Point a Git repository object at the public repo
CREATE OR REPLACE GIT REPOSITORY healthcare_doc_ai_repo
  API_INTEGRATION = github_api_integration
  ORIGIN = 'https://github.com/sfc-gh-aprah/AI-Powered-Document-Intelligence-for-Healthcare.git';

-- 3) Fetch the latest, then copy the PDFs into the stage
ALTER GIT REPOSITORY healthcare_doc_ai_repo FETCH;

COPY FILES INTO @CLINICAL_DOCS_STAGE
  FROM @healthcare_doc_ai_repo/branches/main/sample_documents/;

In [ ]:
%%sql -r dataframe_4
-- Refresh the directory table and confirm the files landed
ALTER STAGE CLINICAL_DOCS_STAGE REFRESH;

SELECT relative_path, size, last_modified
FROM DIRECTORY(@CLINICAL_DOCS_STAGE)
ORDER BY relative_path;

## Segment 2 — Parse with `AI_PARSE_DOCUMENT`

One SQL function reads a PDF with **full layout understanding**. `LAYOUT` mode preserves tables
and structure (the right choice for clinical documents); `OCR` mode is text-only.

**Beat 1:** parse a single document and inspect the layout-faithful markdown — scroll to the
discharge medications table.

In [ ]:
%%sql -r dataframe_5
-- Beat 1: parse one document in LAYOUT mode and view the structured markdown
SELECT AI_PARSE_DOCUMENT(
  TO_FILE('@CLINICAL_DOCS_STAGE', 'discharge_summary_doe_anna.pdf'),
  {'mode': 'LAYOUT'}
):content::string AS parsed_markdown;

**Beat 2:** parse the **whole corpus** into a table — one row per page.
`'page_split': true` splits each document into pages; `LATERAL FLATTEN` unpacks them into rows.

In [ ]:
%%sql -r dataframe_6
-- Beat 2: parse all documents -> one row per page
CREATE OR REPLACE TABLE PARSED_DOCUMENTS AS
SELECT
    d.relative_path                  AS file_name,
    f.value:index::int               AS page_number,
    f.value:content::string          AS page_content
FROM DIRECTORY(@CLINICAL_DOCS_STAGE) d,
     LATERAL FLATTEN(
        input => AI_PARSE_DOCUMENT(
            TO_FILE('@CLINICAL_DOCS_STAGE', d.relative_path),
            {'mode': 'LAYOUT', 'page_split': TRUE}
        ):pages
     ) f;

In [ ]:
%%sql -r dataframe_7
-- Twelve pages, fully parsed, in a governed table
SELECT file_name, page_number, LEFT(page_content, 300) AS page_preview
FROM PARSED_DOCUMENTS
ORDER BY file_name, page_number;

### `AI_EXTRACT` — targeted field extraction with confidence scores

For pulling specific fields (not full text), `AI_EXTRACT` returns a value **and a confidence score**
for each field — the human-in-the-loop hook: auto-accept high-confidence values, route the rest for review.

In [ ]:
%%sql -r dataframe_8
-- Extract specific fields from a fax-format prior auth form
SELECT AI_EXTRACT(
  file => TO_FILE('@CLINICAL_DOCS_STAGE', 'prior_auth_form_knee_replacement.pdf'),
  responseFormat => {
    'cpt_code': 'What is the requested CPT procedure code?',
    'diagnosis': 'What is the primary diagnosis and ICD-10 code?',
    'estimated_cost': 'What is the estimated cost of the procedure?'
  },
  scores => TRUE
) AS extracted;

## Segment 3 — Semantic search with Cortex Search

Clinicians ask questions in clinical language that rarely matches the document's exact words.
**Cortex Search** provides semantic (meaning-based) retrieval. The whole service is one DDL statement —
no embedding model selection, no vector database, no indexing pipeline.

`TARGET_LAG = '1 hour'` is a freshness contract: new rows index themselves within the hour (matters for Segment 5).

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE CORTEX SEARCH SERVICE CLINICAL_DOC_SEARCH
  ON page_content
  ATTRIBUTES file_name, page_number
  WAREHOUSE = HEALTHCARE_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT page_content, file_name, page_number
    FROM PARSED_DOCUMENTS
  );

### Test it from SQL

You can also demo searches live in the **Snowsight playground** (left nav → AI & ML → Cortex Search).
Here we query the service directly with `SEARCH_PREVIEW`. Note the result returns the
**source document and page** for traceability.

In [ ]:
%%sql -r dataframe_10
-- Semantic match: the document never says "fluid overload" or "reduced ejection fraction"
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'CLINICAL_DOC_SEARCH',
    '{
       "query": "patient with fluid overload and reduced ejection fraction",
       "columns": ["file_name", "page_number", "page_content"],
       "limit": 3
     }'
  )
):results AS results;

In [ ]:
%%sql -r dataframe_11
-- The utilization-review question, answered in seconds
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'CLINICAL_DOC_SEARCH',
    '{
       "query": "failed physical therapy and injections before surgery",
       "columns": ["file_name", "page_number", "page_content"],
       "limit": 3
     }'
  )
):results AS results;

## Segment 4 — Snowflake CoWork agent (UI)

This step is done in the Snowsight UI, not in the notebook. Build an agent that uses the
Cortex Search service as a tool, then ask it natural-language questions.

**Create the agent** (Snowsight → AI & ML → Agents → Create agent):
- **Platform integration:** *Create this agent for Snowflake Intelligence*
- **Object name:** `CLINICAL_DOCS_AGENT`
- **Display name:** `Clinical Document Assistant`

**Configure (Edit):**
- **Description:** *Answers questions about clinical documents — discharge summaries, labs, prior authorizations, and progress notes.*
- **Response instruction:**
  > Always cite the source document and page for every clinical fact. If information is not in the documents, say so. Never speculate about patient information.
- **Tools → add Cortex Search →** select `CLINICAL_DOC_SEARCH`.

**Surface it:** Agents → *Snowflake Intelligence* tab → settings → add your agent.
The chat interface runs with each user's own role and default warehouse — RBAC is enforced automatically.

### Demo questions
1. *Summarize Anna Doe's heart failure hospitalization and what changed at her follow-up visit.*
2. *What conservative treatments did Thomas Roe fail before his knee replacement was authorized?*
3. *Which patients are on furosemide, and what's their most recent kidney function?*

> **Privilege note:** creating the search service requires `SNOWFLAKE.CORTEX_USER`; creating the
> agent requires `CREATE AGENT`. Grant the search service to the agent's role with the cell below if needed.

In [ ]:
%%sql -r dataframe_12
-- Optional: let the agent's role use the search service (adjust role name)
GRANT USAGE ON CORTEX SEARCH SERVICE CLINICAL_DOC_SEARCH
  TO ROLE ACCOUNTADMIN;

## Segment 5 — Automate the pipeline

Production = a new document arrives, the pipeline runs, the index updates, nobody touches anything.

A **stored procedure** refreshes the stage and parses only *new* files (incremental by construction),
and a **task** runs it on a schedule. The Cortex Search service refreshes itself within its `TARGET_LAG`.

In [ ]:
%%sql -r dataframe_13
-- Stored procedure: refresh stage + parse only files not already in PARSED_DOCUMENTS
CREATE OR REPLACE PROCEDURE PROCESS_NEW_DOCS_PROC()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN
  ALTER STAGE CLINICAL_DOCS_STAGE REFRESH;

  INSERT INTO PARSED_DOCUMENTS (file_name, page_number, page_content)
  SELECT
      d.relative_path,
      f.value:index::int,
      f.value:content::string
  FROM DIRECTORY(@CLINICAL_DOCS_STAGE) d,
       LATERAL FLATTEN(
          input => AI_PARSE_DOCUMENT(
              TO_FILE('@CLINICAL_DOCS_STAGE', d.relative_path),
              {'mode': 'LAYOUT', 'page_split': TRUE}
          ):pages
       ) f
  WHERE d.relative_path NOT IN (SELECT DISTINCT file_name FROM PARSED_DOCUMENTS);

  RETURN 'Processed new documents.';
END;
$$;

In [ ]:
%%sql -r dataframe_14
-- Native scheduler: hourly, incremental. Created suspended by default.
CREATE OR REPLACE TASK PROCESS_NEW_DOCUMENTS
  WAREHOUSE = HEALTHCARE_WH
  SCHEDULE = 'USING CRON 0 * * * * America/Los_Angeles'
AS
  CALL PROCESS_NEW_DOCS_PROC();

### The moment of truth

Upload the **7th PDF** from the `new_arrival/` folder (`discharge_summary_poe_sarah.pdf`) into
`CLINICAL_DOCS_STAGE` via Snowsight (drag it in to play "the fax machine"). This document was
deliberately kept out of the Segment 1 load, so the pipeline has never seen it.

Then trigger the task manually instead of waiting for the top of the hour.

In [ ]:
%%sql -r dataframe_15
-- Manually run the task once (works even while the task is suspended)
EXECUTE TASK PROCESS_NEW_DOCUMENTS;

In [ ]:
%%sql -r dataframe_16
-- Confirm the new document's pages landed
SELECT file_name, page_number, LEFT(page_content, 200) AS page_preview
FROM PARSED_DOCUMENTS
ORDER BY file_name, page_number;

### The full loop

Document arrives → task parses it → search index absorbs it (within `TARGET_LAG`) →
the agent can answer questions about it. No reindex job, no pipeline glue.

---

#### Cleanup (optional)

In [ ]:
%%sql -r dataframe_17
-- Optional teardown
-- DROP TASK IF EXISTS PROCESS_NEW_DOCUMENTS;
-- DROP PROCEDURE IF EXISTS PROCESS_NEW_DOCS_PROC();
-- DROP CORTEX SEARCH SERVICE IF EXISTS CLINICAL_DOC_SEARCH;
-- DROP TABLE IF EXISTS PARSED_DOCUMENTS;
-- DROP STAGE IF EXISTS CLINICAL_DOCS_STAGE;
-- DROP SCHEMA IF EXISTS HEALTHCARE_DOC_AI.CLINICAL_DOCS;
-- DROP DATABASE IF EXISTS HEALTHCARE_DOC_AI;
SELECT 'Uncomment the statements above to tear down demo objects.' AS note;